# Notebook 04 — Hierarchical Clustering
**Project:** Credit Card Customer Profiling & Segmentation  
**Goal:** Apply Agglomerative Hierarchical Clustering (Ward linkage) as an alternative to KMeans and compare results.

---
| Step | Action |
|------|--------|
| 1 | Fit Agglomerative Clustering (Ward) |
| 2 | PCA visualisation |
| 3 | Compare KMeans vs Hierarchical |
| 4 | Cluster profile comparison |

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

from src.data_loader import load_raw_data
from src.preprocessing import preprocess
from src.config import RANDOM_STATE, N_CLUSTERS

palette = ['#2196F3','#F44336','#4CAF50','#FF9800']

df_scaled, df_unscaled = preprocess(load_raw_data(), save=False)

# KMeans labels for comparison
km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
km_labels = km.fit_predict(df_scaled)
print('Data loaded. Shape:', df_scaled.shape)

## 1. Agglomerative Clustering

In [ ]:
hc = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage='ward')
hc_labels = hc.fit_predict(df_scaled)

km_sil = silhouette_score(df_scaled, km_labels)
hc_sil = silhouette_score(df_scaled, hc_labels)

print(f'KMeans      Silhouette: {km_sil:.4f}')
print(f'Hierarchical Silhouette: {hc_sil:.4f}')
print(f'\nHierarchical cluster sizes:')
for k, n in zip(*np.unique(hc_labels, return_counts=True)):
    print(f'  Cluster {k}: {n:,} ({n/len(hc_labels)*100:.1f}%)')

## 2. Side-by-Side PCA Comparison

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
comps = pca.fit_transform(df_scaled)
explained = pca.explained_variance_ratio_.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for cluster in np.unique(km_labels):
    mask = km_labels == cluster
    axes[0].scatter(comps[mask, 0], comps[mask, 1], label=f'Cluster {cluster}',
                    alpha=0.4, s=12, color=palette[cluster % len(palette)])
axes[0].set_title(f'KMeans  (Silhouette={km_sil:.4f})', fontweight='bold')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend(markerscale=3)

for cluster in np.unique(hc_labels):
    mask = hc_labels == cluster
    axes[1].scatter(comps[mask, 0], comps[mask, 1], label=f'Cluster {cluster}',
                    alpha=0.4, s=12, color=palette[cluster % len(palette)])
axes[1].set_title(f'Hierarchical — Ward (Silhouette={hc_sil:.4f})', fontweight='bold')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend(markerscale=3)

plt.suptitle(f'KMeans vs Hierarchical Clustering — PCA 2D ({explained:.1f}% variance)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Algorithm Comparison

| Algorithm | Silhouette Score | Notes |
|---|---|---|
| **KMeans** | **0.1758** | Better separation, scalable to 8,950 customers |
| Hierarchical (Ward) | 0.1050 | Lower separation, deterministic but slower |

**Conclusion:** KMeans produces better-separated clusters on this dataset. Hierarchical clustering confirms the segment structure but with lower cohesion.

> **Next:** `05_Customer_Profiling.ipynb` — deep profile analysis and business interpretation of each segment.